# Lab 2 — Logistic Regression (Loan Default)

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Prepare feature matrix $X$ and binary target $y$ for classification.
2. Split data with **stratification** to preserve class balance.
3. Fit **`LogisticRegression`** and interpret intercept / coefficients.
4. Compare **`predict_proba`** (probability) vs **`predict`** (hard label).

> **Checkpoints:** train **800** / test **200** · `int_rate` coef positive · sample preds include **0** and **1**



## Logistic regression in one slide

**Linear model on log-odds:**

$$
	ext{logit}(p) = \lnrac{p}{1-p} = eta_0 + eta_1 x_1 + \cdots + eta_p x_p
$$

**Sigmoid converts back to probability:**

$$
p = rac{1}{1 + e^{-(eta_0 + eta_1 x_1 + \cdots)}}
$$

| vs Day 2 Linear Regression | Logistic Regression |
|---------------------------|---------------------|
| Target: continuous rating | Target: 0/1 default |
| Output: any real number | Output: probability in [0, 1] |
| `LinearRegression` | `LogisticRegression` |


---

## 1. Load data and select features


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-03":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
X = df[NUMERIC_FEATURES]
y = df["default"]

print(f"X shape: {X.shape}")
print(f"default rate: {y.mean():.4f}")
display(X.head(3))


---

## 2. Train/test split with stratification

`stratify=y` keeps the same default rate in train and test — important when classes are imbalanced (critical on Day 6).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}")
print(f"test size:  {len(X_test)}")
print(f"default rate (train): {y_train.mean():.4f}")
print(f"default rate (test):  {y_test.mean():.4f}")

assert len(X_train) == 800 and len(X_test) == 200


---

## 3. Fit logistic regression


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Lab 2 — Logistic regression")
print(f"intercept (beta_0): {model.intercept_[0]:.4f}")
for name, coef in zip(NUMERIC_FEATURES, model.coef_[0]):
    print(f"  {name}: {coef:.4f}")


### Coefficient interpretation

- **`int_rate` (+):** Higher interest rate → higher log-odds of default (riskier borrowers).
- **`dti` (+):** Higher debt-to-income → higher default log-odds.
- Coefficients near zero on `loan_amnt` / `annual_inc` in this lab sample — features may be weakly linearly separable alone.

Coefficients are on the **log-odds scale**, not probability directly.


---

## 4. Sigmoid visualization


In [ ]:
x = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-x))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, sigmoid, color="steelblue", lw=2)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.7)
ax.axvline(0, color="gray", linestyle="--", alpha=0.7)
ax.set_xlabel("log-odds (eta)")
ax.set_ylabel("probability p")
ax.set_title("Sigmoid: log-odds → probability")
plt.tight_layout()
plt.show()


At log-odds = 0, probability = **0.5** — the default classification threshold for `predict()`.


---

## 5. `predict_proba` vs `predict`

| Method | Output |
|--------|--------|
| `predict_proba(X)[:, 1]` | P(default=1) per row |
| `predict(X)` | 0 or 1 using threshold **0.5** |


In [ ]:
proba = model.predict_proba(X_test.head(3))[:, 1]
pred = model.predict(X_test.head(3))

comparison = pd.DataFrame({
    "int_rate": X_test.head(3)["int_rate"].values,
    "dti": X_test.head(3)["dti"].values,
    "P(default)": proba.round(4),
    "predicted_label": pred,
    "actual_default": y_test.head(3).values,
})
display(comparison)

print(f"sample P(default): {proba.round(4)}")
print(f"sample predictions: {pred.tolist()}")
assert 0 in pred and 1 in pred


Row with P(default)=**0.685** → predicted **1** (≥ 0.5). Row with **0.355** → predicted **0**.


---

## 6. Manual log-odds for one test row (optional deep dive)


In [ ]:
row = X_test.iloc[0]
eta = model.intercept_[0] + np.dot(model.coef_[0], row.values)
p_manual = 1 / (1 + np.exp(-eta))
p_sklearn = model.predict_proba(row.values.reshape(1, -1))[0, 1]

print(f"log-odds eta: {eta:.4f}")
print(f"manual p:     {p_manual:.4f}")
print(f"sklearn p:    {p_sklearn:.4f}")


---

## 7. Feature vs default (exploratory)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, x="default", y="int_rate", ax=axes[0], palette="Set2")
axes[0].set_title("Interest rate by default")
sns.boxplot(data=df, x="default", y="dti", ax=axes[1], palette="Set2")
axes[1].set_title("DTI by default")
plt.tight_layout()
plt.show()


Defaults tend toward higher `int_rate` and `dti` — consistent with positive coefficients.


---

## 8. Checkpoint summary


In [ ]:
print("=" * 50)
print(f"train: {len(X_train)}, test: {len(X_test)}")
print(f"intercept: {model.intercept_[0]:.4f}")
print(f"int_rate coef: {model.coef_[0][1]:.4f} (expect > 0)")
assert model.coef_[0][1] > 0
assert model.coef_[0][3] > 0  # dti
print("\n✓ Checkpoint assertions passed")


---

## Reflection questions

1. What happens to predictions if you change the threshold from 0.5 to 0.7?
2. Why use `stratify=y` even when classes are nearly balanced?
3. Which metrics evaluate this model best? *(Lab 3 — confusion matrix, Lab 4 — AUC)*

**Previous:** [Lab 1 — Probability](lab01_probability_exercises.ipynb)  
**Next:** [Lab 3 — Confusion matrix](lab03_confusion_matrix.ipynb)
